# Garment Worker Productivity - Preprocessing and Baseline

This notebook prepares the garment productivity dataset for machine learning and creates a simple baseline model for comparison.

## 1. Loading the Dataset

The dataset is loaded again so that this notebook can run independently without relying on the previous data audit notebook.

In [23]:
import pandas as pd

In [24]:
# Loading the garment productivity dataset
df = pd.read_csv("../data/garments_worker_productivity.csv")

df.head()    # first 5 rows

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


## 2. Data Cleaning

Before the data can be used for machine learning, inconsistent values identified during the data audit are cleaned.


### 2.1 Department Labels

The `department` column contains two versions of the `finishing` category because one contains extra whitespace. Here the whitespace is removed so that both values are treated as the same department.

In [25]:
# Removes extra spaces from the department category labels
df["department"] = df["department"].str.strip()

# Checks department categories after cleaning
df["department"].value_counts()


department
sweing       691
finishing    506
Name: count, dtype: int64

### 2.2 Date Column

The `date` column is currently stored as text, therefore  it is converted to a datetime format here so that the dates can be checked properly before deciding how they should be used during modelling.

In [26]:
# Converts the date column from text to a datetime format
df["date"] = pd.to_datetime(df["date"])

# Checks date range in the dataset
df["date"].min(), df["date"].max()

(Timestamp('2015-01-01 00:00:00'), Timestamp('2015-03-11 00:00:00'))

The dataset covers the period from 1 January 2015 to 11 March 2015. The raw date will not be used directly as an input feature but it is kept for now because it may be useful when designing the training and test split.


## 3. Preparing Features and Target Variable

To preapre the dataset, it is separated into input features and the target variable. The target is `actual_productivity` which is the value the model will try to predict.

### 3.1 Preparing the Data for the Train nd Test Split

The dataset is sorted by date so that earlier records can be used for training and later records for testing. The raw `date` column is kept for ordering but is not used directly as a model input.

In [27]:
# Sortsthe dataset by date before creating the training and test sets
df = df.sort_values("date").reset_index(drop=True)

# Separates input features from the target variable
X = df.drop(columns=["actual_productivity", "date"])
y = df["actual_productivity"]

X.shape, y.shape

((1197, 13), (1197,))

### 3.2 Train/Test Split

The data is divided chronologically using whole dates. Approximately the first 80% of dates are used for training and the remaining dates are kept for testing. This prevents records from the same date being split between the training and test sets.

In [28]:
# Finds the unique dates in chronological order
unique_dates = df["date"].sort_values().unique()

# Uses approximately the first 80% of dates for training
split_index = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates[split_index]

# Creatse training and test sets using whole dates
train_mask = df["date"] < cutoff_date
test_mask = df["date"] >= cutoff_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

# Checks the size of each set
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((943, 13), (254, 13), (943,), (254,))

## 4. Preprocessing Pipeline

The input features need to be prepared before they can be used by the machine learning models. Numerical and categorical features are processed separately because they require different preprocessing steps.


### 4.1 Separating numerical and categorical features

In [29]:
# Categorical features
categorical_features = [
    "quarter",
    "department",
    "day",
    "team"
]

# Numerical features
numerical_features = [
    "targeted_productivity",
    "smv",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]

# WIP is handled separately because it has missing values
wip_feature = ["wip"]

### 4.2 The import the preprocessing tools

In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### 4.3 Numerical preprocessing

In [40]:
# Filling missing values using the median and scaling numerical features
numerical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [32]:
# Used zero instead of assigning finishing rows to a typical sewing WIP value
wip_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

### 4.4 Categorical preprocessing

In [33]:
# Converts categorical values into numerical columns
categorical_preprocessor = OneHotEncoder(
    handle_unknown="ignore"
)

### 4.5 Combininmg numerical and cateogrical

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_preprocessor, numerical_features),
        ("wip", wip_preprocessor, wip_feature),
        ("cat", categorical_preprocessor, categorical_features)
    ]
)

## 5. Baseline Model

A DummyRegressor is used as a simple baseline model in my project. It predicts the average productivity value from the training data rather than learning relationships between the input features. This provides a basic result that later machine learning models should aim to improve on. The preprocessing pipeline defined above will be used with the learned models.

In [35]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [41]:
# Creates a simple mean-based baseline model
baseline_model = DummyRegressor(strategy="mean")

In [37]:
# Fitting the baseline model using the training data
baseline_model.fit(X_train, y_train)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'mean'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[0.74]]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](13,)","['quarter','department','day',...,'idle_men','no_of_style_change', 'no_of_workers']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,13
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [38]:
# Makes predictions for the unseen test data
y_pred_baseline = baseline_model.predict(X_test)

In [39]:
# Evaluates the baseline predictions
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = root_mean_squared_error(y_test, y_pred_baseline)
baseline_r2 = r2_score(y_test, y_pred_baseline)

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Baseline R²: {baseline_r2:.4f}")

Baseline MAE: 0.1216
Baseline RMSE: 0.1659
Baseline R²: -0.0515


### Baseline Observations and Results

The baseline model produced an MAE of 0.1216 and an RMSE of 0.1659. This means that the baseline predictions differ from the actual productivity values by approximately 0.12 on average with some larger errors reflected in the RMSE.  

The R² value is -0.0515. This shows that the simple mean-based baseline does not explain the variation in productivity in the later test data. This is expected from a basic baseline model and gives the later learned models a result to improve on.